### Pacotes importados

In [1]:
using LinearAlgebra
using Printf


## Chapter 10: Newton's local method

### Algorithm 10.1: Newton's local method

![image.png](attachment:09eb2ad4-da2e-4714-9c4e-099da89b43c7.png)

![image.png](attachment:0c619c47-042c-45a2-ba7f-c0b81e685d39.png)

Example 5.8: $f(x_1,x_2) = \frac{1}{2} x_1^2 + x_1 \cos(x_2)$; $x_0 = [1.0, 1.0]$

In [2]:
# Example 5.8
# f(x1,x2) = 1/2*x1^2 + x1*cos(x2)

function f_ex58(x)
    return 0.5*x[1]^2 + x[1]*cos(x[2])
end

function grad_ex58(x)
    return [
        x[1] + cos(x[2]),
        -x[1]*sin(x[2])
    ]
end

function hess_ex58(x)
    return [
        1.0              -sin(x[2]);
        -sin(x[2])       -x[1]*cos(x[2])
    ]
end

function newton_local(f, grad_f, hess_f, x0; eps=1e-8, max_iter=50)
    x = copy(x0)

    println("Newton local method")
    println("k | x | f(x) | ||grad||")

    for k in 0:max_iter
        g = grad_f(x)
        fx = f(x)

        @printf("%2d | [% .6f, % .6f] | %.8f | %.8e\n", k, x[1], x[2], fx, norm(g))

        if norm(g) < eps
            return x, fx, k
        end

        H = hess_f(x)
        dx = -H \ g
        x = x + dx
    end

    return x, f(x), max_iter
end

x0 = [1.0, 1.0]
x_star, f_star, iters = newton_local(f_ex58, grad_ex58, hess_ex58, x0)

println("\nFinal result")
println("x* = ", round.(x_star, digits=8))
println("f(x*) = ", round(f_star, digits=8))
println("iterations = ", iters)


Newton local method
k | x | f(x) | ||grad||
 0 | [ 1.000000,  1.000000] | 1.04030231 | 1.75516512e+00
 1 | [-0.233845,  1.364192] | -0.02062862 | 2.30665382e-01
 2 | [ 0.010814,  1.584836] | -0.00009335 | 1.12840545e-02
 3 | [-0.000002,  1.570793] | -0.00000000 | 2.32349802e-06
 4 | [ 0.000000,  1.570796] | 0.00000000 | 8.35430835e-17

Final result
x* = [0.0, 1.57079633]
f(x*) = 0.0
iterations = 4


### Algorithm 10.2: Newton's local method by quadratic modeling

![image.png](attachment:1129351f-7e94-42b8-bd28-f2a73dd0da69.png)

We test the algorithm on the Rosenbrock function. It is described in Section 11.6 of the book for two variables.  The implementation below involves $n$ variables: \\[f(x) = \sum_{i=1}^{n-1} f_i(x) = \sum_{i=1}^{n-1} 100 (x_{i+1}-x_i^2)^2 + (1-x_i)^2.\\]
The calculation of the derivatives is based on the partial derivatives of the functions $f_i$:
\\[
\begin{array}{rcl}
\partial f_i/\partial x_i&=& -400 x_i (x_{i+1}-x_i^2) - 2(1-x_i), \\\\
\partial f_i/\partial x_{i+1}&=& 200 (x_{i+1}-x_i^2), \\\\
\partial^2 f_i/\partial x^2_i  &=& -400 x_{i+1}+ 1200 x_i^2 + 2, \\\\
\partial^2 f_i/\partial x_i \partial x_{i+1}  &=& -400 x_i^2, \\\\
\partial^2 f_i/\partial x^2_{i+1}  &=& 200.
\end{array}
\\]

In [3]:
# Newton's local method by quadratic modeling
# Tested on the n-dimensional Rosenbrock function

function rosenbrock(x)
    n = length(x)
    return sum(100*(x[i+1] - x[i]^2)^2 + (1 - x[i])^2 for i in 1:n-1)
end

function grad_rosenbrock(x)
    n = length(x)
    g = zeros(n)

    for i in 1:n-1
        g[i] += -400*x[i]*(x[i+1] - x[i]^2) - 2*(1 - x[i])
        g[i+1] += 200*(x[i+1] - x[i]^2)
    end

    return g
end

function hess_rosenbrock(x)
    n = length(x)
    H = zeros(n, n)

    for i in 1:n-1
        H[i,i] += -400*x[i+1] + 1200*x[i]^2 + 2
        H[i,i+1] += -400*x[i]
        H[i+1,i] += -400*x[i]
        H[i+1,i+1] += 200
    end

    return H
end

function cg_spd(A, b; tol=1e-10, max_iter=1000)
    x = zeros(length(b))
    r = b - A*x
    p = copy(r)
    rsold = dot(r, r)

    for k in 1:max_iter
        Ap = A*p
        αcg = rsold / dot(p, Ap)
        x = x + αcg*p
        r = r - αcg*Ap
        rsnew = dot(r, r)

        if sqrt(rsnew) < tol
            break
        end

        p = r + (rsnew/rsold)*p
        rsold = rsnew
    end

    return x
end

function newton_quadratic_model(f, grad_f, hess_f, x0; eps=1e-8, max_iter=100, method=:direct)
    x = copy(x0)

    println("Newton by quadratic modeling - method = ", method)
    println("k | f(x) | ||grad|| | x")

    for k in 0:max_iter
        g = grad_f(x)
        H = hess_f(x)
        fx = f(x)

        @printf("%2d | %.10f | %.8e | %s\n", k, fx, norm(g), string(round.(x, digits=6)))

        if norm(g) < eps
            return x, fx, k
        end

        if method == :direct
            # Cholesky is used because the quadratic model requires positive definite Hessian.
            F = cholesky(Symmetric(H))
            dx = F \ (-g)
        elseif method == :cg
            # Also requires positive definite Hessian.
            cholesky(Symmetric(H))
            dx = cg_spd(H, -g)
        else
            error("Unknown method. Use :direct or :cg.")
        end

        x = x + dx
    end

    return x, f(x), max_iter
end

# Rosenbrock test
x0_ros = [-1.2, 1.0]
x_star_ros, f_star_ros, iters_ros = newton_quadratic_model(
    rosenbrock,
    grad_rosenbrock,
    hess_rosenbrock,
    x0_ros,
    method=:direct
)

println("\nFinal result - Rosenbrock")
println("x* = ", round.(x_star_ros, digits=8))
println("f(x*) = ", round(f_star_ros, digits=8))
println("iterations = ", iters_ros)


Newton by quadratic modeling - method = direct
k | f(x) | ||grad|| | x
 0 | 24.2000000000 | 2.32867688e+02 | [-1.2, 1.0]
 1 | 4.7318843253 | 4.63942621e+00 | [-1.175281, 1.380674]
 2 | 1411.8451793103 | 1.37078985e+03 | [0.763115, -3.175034]
 3 | 0.0559655168 | 4.73110379e-01 | [0.76343, 0.582825]
 4 | 0.3131890761 | 2.50274456e+01 | [0.999995, 0.944027]
 5 | 0.0000000000 | 8.60863352e-06 | [0.999996, 0.999991]
 6 | 0.0000000000 | 8.28570579e-09 | [1.0, 1.0]

Final result - Rosenbrock
x* = [1.0, 1.0]
f(x*) = 0.0
iterations = 6


We now apply the algorithm on example 5.8. In this case, the algorithm fails to converge, and one hessian is not positive definite. We try first using the direct method to solve the quadratic problem. An error is triggered.

In [4]:
# Applying quadratic modeling to Example 5.8 using the direct method.
# This fails because the Hessian is not positive definite at the starting point.

x0 = [1.0, 1.0]

try
    x_star_direct, f_star_direct, iters_direct = newton_quadratic_model(
        f_ex58,
        grad_ex58,
        hess_ex58,
        x0,
        method=:direct
    )

    println("x* = ", round.(x_star_direct, digits=8))
    println("f(x*) = ", round(f_star_direct, digits=8))
catch err
    println("Direct method failed.")
    println("Reason: the Hessian is not positive definite at some iteration.")
    println("Julia error: ", err)
end


Newton by quadratic modeling - method = direct
k | f(x) | ||grad|| | x
 0 | 1.0403023059 | 1.75516512e+00 | [1.0, 1.0]
Direct method failed.
Reason: the Hessian is not positive definite at some iteration.
Julia error: PosDefException(2)


If we try with the conjugate gradient method, an error is also triggered.

In [5]:
# Applying quadratic modeling to Example 5.8 using conjugate gradient.
# This also fails because CG for the quadratic model requires a positive definite Hessian.

x0 = [1.0, 1.0]

try
    x_star_cg, f_star_cg, iters_cg = newton_quadratic_model(
        f_ex58,
        grad_ex58,
        hess_ex58,
        x0,
        method=:cg
    )

    println("x* = ", round.(x_star_cg, digits=8))
    println("f(x*) = ", round(f_star_cg, digits=8))
catch err
    println("Conjugate gradient method failed.")
    println("Reason: the Hessian is not positive definite at some iteration.")
    println("Julia error: ", err)
end


Newton by quadratic modeling - method = cg
k | f(x) | ||grad|| | x
 0 | 1.0403023059 | 1.75516512e+00 | [1.0, 1.0]
Conjugate gradient method failed.
Reason: the Hessian is not positive definite at some iteration.
Julia error: PosDefException(2)
